
# Module 4: Unity Catalog ガバナンス（45分 ★）

## 目的
- 粗→細のアクセス制御を実演
- 行フィルタ・列マスキング・タグ付け・ ABAC
- 「捨てない・重ねない・コピーしない」= データを動かさず in-place にガバナンスを重ねる

## FE 制約
- RLS/マスキング/ABAC はサーバレスで動作 ✓
- SCIM/SSO/アカウントポリシーは FE 不可 → ワークスペース内グループ + GRANT で概念実演

In [0]:
%run ./config


## Step 1: GRANT による粗→細のアクセス制御

UC の権限モデル:
- **Catalog レベル**: カタログ全体へのアクセス
- **Schema レベル**: スキーマ内全テーブルへのアクセス
- **Table レベル**: 個別テーブルへのアクセス

In [0]:
# グループ作成（FE ではワークスペース内グループで概念実演）
# 注: FE では SCIM/SSO が使えないため、アカウントグループ作成は口頭説明
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# GRANT の例（グループがあれば実行可能）
# 粗い権限: Schema 全体への読み取り
print("""
-- 粗い権限の例:
GRANT USE CATALOG ON CATALOG <catalog> TO `data_readers`;
GRANT USE SCHEMA ON SCHEMA <catalog>.<schema> TO `data_readers`;
GRANT SELECT ON SCHEMA <catalog>.<schema> TO `data_readers`;

-- 細かい権限の例:
GRANT SELECT ON TABLE <catalog>.<schema>.vehicle_logs TO `analyst_team`;
""")


## Step 2: 行フィルタ関数（Row-Level Security）

ユーザーの属性に応じてデータをフィルタリングします。

In [0]:
# 行フィルタ関数の作成
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{USER_SCHEMA}.row_filter_by_location(location_val STRING)
RETURNS BOOLEAN
RETURN 
  -- 管理者は全データ参照可、それ以外は東京のみ
  IS_ACCOUNT_GROUP_MEMBER('admins') OR location_val = '東京'
""")

print("✅ 行フィルタ関数作成完了")

In [0]:
# 行フィルタをテーブルに適用
spark.sql(f"""
ALTER TABLE {CATALOG}.{USER_SCHEMA}.vehicle_logs
SET ROW FILTER {CATALOG}.{USER_SCHEMA}.row_filter_by_location ON (location)
""")

print("✅ 行フィルタ適用完了")
print("   admins グループ以外は「東京」のデータのみ表示されます")

In [0]:
# 行フィルタの効果確認
display(spark.sql(f"""
  SELECT location, COUNT(*) AS cnt
  FROM {CATALOG}.{USER_SCHEMA}.vehicle_logs
  GROUP BY location
"""))


## Step 3: 列マスキング

機密カラムの値を隠します。

In [0]:
# 列マスキング関数の作成
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{USER_SCHEMA}.mask_vehicle_id(vehicle_id_val STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_ACCOUNT_GROUP_MEMBER('admins') THEN vehicle_id_val
    ELSE CONCAT('VH-***')
  END
""")

print("✅ 列マスキング関数作成完了")

In [0]:
# 列マスキングをテーブルに適用
spark.sql(f"""
ALTER TABLE {CATALOG}.{USER_SCHEMA}.vehicle_logs
ALTER COLUMN vehicle_id SET MASK {CATALOG}.{USER_SCHEMA}.mask_vehicle_id
""")

print("✅ 列マスキング適用完了")
print("   admins グループ以外は vehicle_id が 'VH-***' と表示されます")

In [0]:
# マスキングの効果確認
display(spark.sql(f"SELECT log_id, vehicle_id, location, battery_pct FROM {CATALOG}.{USER_SCHEMA}.vehicle_logs LIMIT 5"))


## Step 4: タグ付けと ABAC ポリシー

タグでデータを分類し、属性ベースのアクセス制御（ABAC）を実現します。

In [0]:
# テーブル・カラムにタグを付与
spark.sql(f"""
ALTER TABLE {CATALOG}.{USER_SCHEMA}.vehicle_logs
SET TAGS ('sensitivity' = 'medium', 'domain' = 'operations')
""")

spark.sql(f"""
ALTER TABLE {CATALOG}.{USER_SCHEMA}.vehicle_logs
ALTER COLUMN vehicle_id SET TAGS ('pii' = 'name')
""")

spark.sql(f"""
ALTER TABLE {CATALOG}.{USER_SCHEMA}.vehicle_logs
ALTER COLUMN location SET TAGS ('pii' = 'address')
""")

print("✅ タグ付与完了")
print("   - テーブル: sensitivity=medium, domain=operations")
print("   - vehicle_id: pii=name, location: pii=address")

In [0]:
# タグの確認
display(spark.sql(f"""
  SELECT * FROM {CATALOG}.information_schema.table_tags
  WHERE schema_name = '{USER_SCHEMA}'
"""))

In [0]:
display(spark.sql(f"""
  SELECT * FROM {CATALOG}.information_schema.column_tags
  WHERE schema_name = '{USER_SCHEMA}' AND table_name = 'vehicle_logs'
"""))


### ABAC ポリシーの説明

ABAC (Attribute-Based Access Control) では、タグに基づいてアクセスポリシーを自動適用できます:

```sql
-- ABAC ポリシーの例（Catalog Explorer から設定）:
-- 「pii=true タグが付いた列は、pii_readers グループのみ参照可」
```

→ Catalog Explorer > テーブル > Governance タブから設定可能


## Step 5: リネージ確認

Catalog Explorer でテーブルのリネージ（データの来歴）を確認できます。

**確認方法:**
1. Catalog Explorer を開く
2. `<catalog>.<schema>.vehicle_logs` を選択
3. **Lineage** タブでデータの流れを確認


## Step 6: 「利用者側/基盤側」の責任分界

| 層 | 責任 | UC での実現 |
| --- | --- | --- |
| テーブルオーナー | データ品質・コメント・タグ付け | COMMENT / SET TAGS |
| 基盤管理者 | アクセスポリシー・監査 | RLS / マスク / ABAC / リネージ |
| 利用者 | GRANT された範囲で利用 | SELECT / USE |

### 「捨てない・重ねない・コピーしない」

- データを動かさず **in-place** にガバナンスを重ねる
- 同じテーブルに対し、ユーザーの属性に応じて見え方が変わる
- コピーを作らないのでデータ拡散リスクがない


## クリーンアップ（任意）

ハンズオン後に行フィルタ・マスキングを削除する場合:

In [0]:
# -- クリーンアップ用（必要な場合のみ実行）
# spark.sql(f"ALTER TABLE {CATALOG}.{USER_SCHEMA}.vehicle_logs DROP ROW FILTER")
# spark.sql(f"ALTER TABLE {CATALOG}.{USER_SCHEMA}.vehicle_logs ALTER COLUMN vehicle_id DROP MASK")
# print("✅ クリーンアップ完了")


## ✅ 完了条件

- [x] 同一テーブルで権限によって見え方（マスクあり/なし、行フィルタ等）が変わることを示せた
- [x] タグ付けと ABAC の概念を理解した
- [x] 「in-place ガバナンス」の考え方を理解した